In [ ]:
!pip install gensim
!pip install opendatasets
!python -m spacy download en_core_web_lg


In [ ]:
!pip install -q transformers datasets evaluate accelerate -U

In [ ]:
!pip install --upgrade --force-reinstall pyarrow

In [ ]:
import opendatasets as od
import pandas as pd
import numpy as np
import re
import string
import evaluate
import torch
import spacy
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset


model_id = "roupenminassian/TwHIN-BERT-Misinformation-Classifier"
tqdm.pandas()

In [ ]:
#KGAT_9e22a898763110e4579177643e19528b
od.download("https://www.kaggle.com/datasets/doanquanvietnamca/liar-dataset/")
od.download("https://www.kaggle.com/datasets/studymart/welfake-dataset-for-fake-news/data")

In [ ]:
df = pd.read_csv("/content/welfake-dataset-for-fake-news/WELFake_Dataset.csv")
df = df.dropna()

df.head()

In [ ]:
df["context"] = df["title"] + " " + df["text"]
df = df.drop(columns=["title", "text"])

In [ ]:
df_small, _ = train_test_split(
    df,
    train_size=0.05,
    stratify=df['label'],
    random_state=393
)

In [ ]:
nlp = spacy.load("en_core_web_lg")

def process_text(
    text: str,
    mask_types: set = {"PERSON", "ORG", "GPE", "LOC", "NORP", "FAC", "TIME", "DATE"},
    max_words: int = 300
) -> str:

    words = str(text).split()
    text = " ".join(words[:max_words])

    # URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Mask named entities and delete them
    doc = nlp(text[:100_000])
    for ent in reversed(doc.ents):
        if ent.label_ in mask_types:
            text = text[:ent.start_char] + text[ent.end_char:]

    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df_small['context_processed'] = df_small['context'].progress_apply(process_text)

In [ ]:
df_train, df_val = train_test_split(
    df_small,
    test_size=0.2,
    stratify=df_small['label'],
    random_state=393
)

In [ ]:
# Word2Vec training

tokenized_docs = [text.split() for text in df_train['context_processed']]
val_tokens = [text.split() for text in df_val['context_processed']]

w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=300,
    window=5,
    min_count=3,
    workers=4,
    epochs=10
)

def get_custom_doc_vector(tokens, model):
    valid_tokens = [word for word in tokens if word in model.wv.key_to_index]
    if not valid_tokens:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_tokens], axis=0)

X_train_w2v = np.vstack([get_custom_doc_vector(tokens, w2v_model) for tokens in tokenized_docs])
X_val_w2v = np.vstack([get_custom_doc_vector(tokens, w2v_model) for tokens in val_tokens])

lr_custom = LogisticRegression(max_iter=1000)
lr_custom.fit(X_train_w2v, df_train['label'])

scores_custom = cross_val_score(lr_custom, X_train_w2v, df_train['label'], cv=5, scoring='accuracy')
print(f"Word2Vec + LR Accuracy: {scores_custom.mean():.4f} ± {scores_custom.std():.4f}")


In [ ]:
#RF
y_train = df_train['label'].values
y_val = df_val['label'].values

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = tfidf.fit_transform(df_train['context_processed'])

X_val_tfidf = tfidf.transform(df_val['context_processed'])

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, df_train['label'])

print("\n[RF WELFake Validation Results]")
rf_val_preds = rf_model.predict(X_val_tfidf)
rf_val_probs = rf_model.predict_proba(X_val_tfidf)[:, 1]

print(f"WELFake RF Accuracy: {accuracy_score(y_val, rf_val_preds):.4f}")
print(f"WELFake RF ROC-AUC: {roc_auc_score(y_val, rf_val_probs):.4f}\n")
print("Classification Report:")
print(classification_report(y_val, rf_val_preds, target_names=["Real (0)", "Fake (1)"]))



In [ ]:
# Embedding matrix setup

vocab_size = len(w2v_model.wv.key_to_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))
token2idx = {}

for word, idx in w2v_model.wv.key_to_index.items():
    new_idx = idx + 1
    embedding_matrix[new_idx] = w2v_model.wv[word]
    token2idx[word] = new_idx

# Vectorize and Pad
X_train_seqs = [[token2idx[token] for token in tokens if token in token2idx] for tokens in tokenized_docs]
X_val_seqs = [[token2idx[token] for token in tokens if token in token2idx] for tokens in val_tokens]

y_train = df_train['label'].values
y_val = df_val['label'].values

train_data = [(seq, label) for seq, label in zip(X_train_seqs, y_train) if len(seq) > 0]
val_data   = [(seq, label) for seq, label in zip(X_val_seqs,   y_val)   if len(seq) > 0]

X_train_seqs, y_train = zip(*train_data)
X_val_seqs,   y_val   = zip(*val_data)

MAX_LEN = 300
X_train_pad = pad_sequences(X_train_seqs, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seqs, maxlen=MAX_LEN, padding='post', truncating='post')

y_train = np.array(y_train)
y_val   = np.array(y_val)

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=False,
        mask_zero=True
    ),
    Bidirectional(LSTM(64)),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=32
)

In [ ]:
loss, accuracy = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"Welfake LSTM Loss: {loss:.4f}")
print(f"Welfake LSTM Accuracy: {accuracy:.4f}")

y_val_probs = model.predict(X_val_pad, verbose=0)
y_val_classes = (y_val_probs >= 0.5).astype(int).flatten()
print(f"LSTM ROC-AUC:{roc_auc_score(y_val, y_val_probs):.4f}\n")

print("Welfake Classification LSTM")
print(classification_report(y_val, y_val_classes, target_names=["Real (0)", "Fake (1)"]))

print("Welfake Native Confusion Matrix LSTM")
cm = confusion_matrix(y_val, y_val_classes)
print(f"TP:  {cm[0][0]:<5} | FN:  {cm[0][1]}")
print(f"FP: {cm[1][0]:<5} | TN:  {cm[1][1]}\n")


lr_val_preds = lr_custom.predict(X_val_w2v)
lr_val_acc = accuracy_score(y_val, lr_val_preds)
print(f"Welfake LR Accuracy: {lr_val_acc:.4f}\n")
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_val, lr_val_preds):.4f}")
print("Welfake Classification LR")
print(classification_report(y_val, lr_val_preds, target_names=['Real', 'Fake']))


In [ ]:
df_liar = pd.read_csv("/content/liar-dataset/test.tsv",
    sep="\t",
    header=None)

In [ ]:
df_liar.head()

In [ ]:
df_liar = df_liar.drop(df_liar.columns[0], axis=1)
df_liar = df_liar.drop(df_liar.columns[2:], axis=1)
df_liar.columns = ["label_text", "context"]

fake_labels = ["false", "pants-fire", "barely-true"]
real_labels = ["half-true", "mostly-true", "true"]

df_liar["label"] = df_liar["label_text"].apply(lambda x: 1 if x in fake_labels else 0)
df_liar = df_liar.drop(columns=["label_text"], axis=1)

df_liar_small, _ = train_test_split(
    df_liar,
    train_size=0.25,
    stratify=df_liar['label'],
    random_state=393
)

In [ ]:
df_liar_small['context_processed'] = df_liar_small['context'].progress_apply(process_text)

In [ ]:
X_liar_tfidf = tfidf.transform(df_liar_small['context_processed'])
y_new_liar = df_liar_small['label'].values

print("\n[RF LIAR Cross-Domain Results]")
rf_liar_preds = rf_model.predict(X_liar_tfidf)
rf_liar_probs = rf_model.predict_proba(X_liar_tfidf)[:, 1]

print(f"LIAR RF Accuracy: {accuracy_score(y_new_liar, rf_liar_preds):.4f}")
print(f"LIAR RF ROC-AUC: {roc_auc_score(y_new_liar, rf_liar_probs):.4f}\n")
print("Classification Report:")
print(classification_report(y_new_liar, rf_liar_preds, target_names=["Real (0)", "Fake (1)"]))

In [ ]:
new_tokens_liar = [text.split() for text in df_liar_small['context_processed']]
y_new_liar = df_liar_small['label'].values

X_liar_seqs = [[token2idx[token] for token in tokens if token in token2idx] for tokens in new_tokens_liar]
X_liar_pad = pad_sequences(X_liar_seqs, maxlen=300, padding='post', truncating='post')
X_liar_w2v = np.vstack([get_custom_doc_vector(tokens, w2v_model) for tokens in new_tokens_liar])



lstm_liar_loss, lstm_liar_acc = model.evaluate(X_liar_pad, y_new_liar, verbose=0)
lstm_liar_probs = model.predict(X_liar_pad, verbose=0)
lstm_liar_preds = (lstm_liar_probs >= 0.5).astype(int).flatten()
print(f"LSTM Cross-Domain Accuracy LIAR: {lstm_liar_acc:.4f}")
print(f"LSTM ROC-AUC:{roc_auc_score(y_new_liar, lstm_liar_probs):.4f}\n")

print("LSTM Classification Report]")
print(classification_report(y_new_liar, lstm_liar_preds, target_names=["Real (0)", "Fake (1)"]))

lr_liar_preds = lr_custom.predict(X_liar_w2v)
lr_liar_acc = accuracy_score(y_new_liar, lr_liar_preds)
print(f"Logistic Regression Cross-Domain Accuracy LIAR: {lr_liar_acc:.4f}\n")
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_new_liar, lr_liar_preds ):.4f}\n")
print("Classification Report:")
print(classification_report(y_new_liar, lr_liar_preds, target_names=["Real (0)", "Fake (1)"]))




In [ ]:
# Copy raw text with all the entities
df_train_bert = df_train[['context', 'label']].copy()
df_train_bert.columns = ['text', 'labels']

df_val_bert = df_val[['context', 'label']].copy()
df_val_bert.columns = ['text', 'labels']

df_liar_bert = df_liar_small[['context', 'label']].copy()
df_liar_bert.columns = ['text', 'labels']

In [ ]:
# Convert to HG datasets
ds_train = Dataset.from_pandas(df_train_bert, preserve_index=False)
ds_val   = Dataset.from_pandas(df_val_bert, preserve_index=False)
ds_liar  = Dataset.from_pandas(df_liar_bert, preserve_index=False)

In [ ]:
# Tokenizer for the model
model_id = "roupenminassian/TwHIN-BERT-Misinformation-Classifier"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

print("Tokenizing datasets...")
tokenized_train = ds_train.map(tokenize_function, batched=True)
tokenized_val   = ds_val.map(tokenize_function, batched=True)
tokenized_liar  = ds_liar.map(tokenize_function, batched=True)

In [ ]:
# Setup
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2, ignore_mismatched_sizes=True)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Fine tune
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Starting Fine-Tuning...")
trainer.train()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/models/twhin-bert-finetuned"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

save_path = "/content/drive/MyDrive/models/twhin-bert-finetuned"

model     = AutoModelForSequenceClassification.from_pretrained(save_path)
tokenizer = AutoTokenizer.from_pretrained(save_path)

In [ ]:
print("Welfake Eval")
val_results = trainer.predict(tokenized_val)
val_preds = np.argmax(val_results.predictions, axis=-1)

val_probs = torch.nn.functional.softmax(torch.tensor(val_results.predictions), dim=-1).numpy()[:, 1]

print(f"WELFake Validation Accuracy: {accuracy_score(tokenized_val['labels'], val_preds):.4f}")
print("Classification Report:")
print(classification_report(tokenized_val['labels'], val_preds, target_names=["Real (0)", "Fake (1)"]))

In [ ]:
print("LIAR Eval")
liar_results = trainer.predict(tokenized_liar)
liar_preds = np.argmax(liar_results.predictions, axis=-1)

# Softmax to the logits to get ROC-AUC
liar_probs = torch.nn.functional.softmax(torch.tensor(liar_results.predictions), dim=-1).numpy()[:, 1]

print(f"LIAR Cross-Domain Accuracy: {accuracy_score(tokenized_liar['labels'], liar_preds):.4f}")
print(f"LIAR ROC-AUC: {roc_auc_score(tokenized_liar['labels'], liar_probs):.4f}")
print("Classification Report:")
print(classification_report(tokenized_liar['labels'], liar_preds, target_names=["Real (0)", "Fake (1)"]))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


data = {
    "Model": ["Logistic Regression", "Random Forest", "LSTM", "BERT"],

    "WELFake Accuracy": [
        accuracy_score(y_val, lr_val_preds),
        accuracy_score(y_val, rf_val_preds),
        accuracy,
        accuracy_score(tokenized_val['labels'], val_preds)
    ],
    "LIAR Accuracy": [
        accuracy_score(y_new_liar, lr_liar_preds),
        accuracy_score(y_new_liar, rf_liar_preds),
        lstm_liar_acc,
        accuracy_score(tokenized_liar['labels'], liar_preds)
    ],

    "WELFake ROC-AUC": [
        roc_auc_score(y_val, lr_val_preds),
        roc_auc_score(y_val, rf_val_probs),
        roc_auc_score(y_val, y_val_probs),
        roc_auc_score(tokenized_val['labels'], val_probs)
    ],
    "LIAR ROC-AUC": [
        roc_auc_score(y_new_liar, lr_liar_preds),
        roc_auc_score(y_new_liar, rf_liar_probs),
        roc_auc_score(y_new_liar, lstm_liar_probs),
        roc_auc_score(tokenized_liar['labels'], liar_probs)
    ]
}

df_results = pd.DataFrame(data)

In [ ]:
df_acc = df_results.melt(id_vars="Model", value_vars=["WELFake Accuracy", "LIAR Accuracy"],
                         var_name="Dataset", value_name="Accuracy")

df_roc = df_results.melt(id_vars="Model", value_vars=["WELFake ROC-AUC", "LIAR ROC-AUC"],
                         var_name="Dataset", value_name="ROC-AUC")

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

ax1 = sns.barplot(data=df_acc, x="Model", y="Accuracy", hue="Dataset", palette=["#2ecc71", "#e74c3c"])
plt.title("The Domain Shift: Model Accuracy (WELFake vs. LIAR)", fontsize=16, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel("Accuracy Score", fontsize=12)
plt.xlabel("Model Architecture", fontsize=12)
plt.legend(title="Testing Dataset", loc="upper left")

for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.3f}",
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                 textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

ax2 = sns.barplot(data=df_roc, x="Model", y="ROC-AUC", hue="Dataset", palette=["#3498db", "#e67e22"])
plt.title("Ranking Performance: ROC-AUC (WELFake vs. LIAR)", fontsize=16, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel("ROC-AUC Score", fontsize=12)
plt.xlabel("Model Architecture", fontsize=12)
plt.legend(title="Testing Dataset", loc="upper left")

for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.3f}",
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                 textcoords='offset points')

plt.axhline(0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
plt.text(-0.4, 0.52, 'Random Guessing Baseline (0.5)', color='gray', fontsize=10)

plt.tight_layout()
plt.show()